This script will take the tif files and turn to parquet which will be used to predict on 

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = Path("/explore/nobackup/people/spotter5/anna_v/v2/predictors/abcfluxmodelv2")
OUT_DIR  = Path("/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors")
os.makedirs(OUT_DIR, exist_ok=True)

STUDY_SHP = "/explore/nobackup/people/spotter5/anna_v/v2/studydomain/studydomain_3413_combined.shp"

DATASETS = [
    {"name": "ALT",            "subdir": "ALT_tif",                                     "kind": "alt_annual"},
    {"name": "ERA5",           "subdir": "ERA5",                                        "kind": "era5_month_code"},
    {"name": "SMAP_L4",        "subdir": "L4_SM_NRv11-4_40N+_soil_moisture_tif",        "kind": "YxxxxMxx"},
    {"name": "LAI_FPAR",       "subdir": "MCD15A3H_lai_fpar",                           "kind": "simple_YYYY_MM"},
    {"name": "LST",            "subdir": "LST",                                         "kind": "simple_YYYY_MM"},
    {"name": "MODIS_AllBands", "subdir": "MOD13A3_MYD13A3",                             "kind": "era5_month_code"},
    {"name": "TerraClimate",   "subdir": "TerraClimate",                                "kind": "simple_YYYY_MM"},
    {"name": "HiHydroSoil",    "subdir": "HiHydroSoil",                                 "kind": "static"},
    {"name": "MERIT_DEM_TPI",  "subdir": "MERIT_DEM_TPI",                               "kind": "static"},
]

# ============================================================
# LOAD STUDY DOMAIN (EPSG:3413)
# ============================================================

study = gpd.read_file(STUDY_SHP)
if study.crs is None or study.crs.to_epsg() != 3413:
    study = study.to_crs(3413)

# ============================================================
# HELPERS
# ============================================================

def normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s.strip())
    return s.strip("_") or ""

def get_band_names(ds: rio.DatasetReader, dataset_name: str):
    descs = ds.descriptions
    names = []
    for i in range(1, ds.count + 1):
        d = descs[i-1] if descs and descs[i-1] else f"band{i}"
        d_norm = normalize_name(d)
        names.append(d_norm if d_norm else f"band{i}")
    return names

def extract_year_month(fname: str, kind: str):
    if kind == "alt_annual":
        m = re.match(r"ALT_(\d{4})(?:_grid)?\.tif$", fname)
        return (int(m.group(1)), None) if m else (None, None)

    if kind == "era5_month_code":
        m = re.match(r".*_(\d{4})_(\d{2})\d{10}-.*\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "YxxxxMxx":
        m = re.match(r".*_Y(\d{4})M(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "simple_YYYY_MM":
        m = re.match(r".*_(\d{4})_(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    return (None, None)

def clip_and_flatten(ds, study_gdf, band_names, round_decimals=3):
    # Reproject study area to match raster CRS
    if ds.crs != study_gdf.crs:
        study_ds = study_gdf.to_crs(ds.crs)
    else:
        study_ds = study_gdf

    shapes = list(study_ds.geometry)

    masked_arr, out_transform = mask(ds, shapes, crop=True, filled=False)

    data = masked_arr.data.astype("float32")
    mask_arr = np.ma.getmaskarray(masked_arr)
    data[mask_arr] = np.nan

    bands, H, W = data.shape
    rows, cols = np.arange(H), np.arange(W)
    rgrid, cgrid = np.meshgrid(rows, cols, indexing="ij")
    xs, ys = rio.transform.xy(out_transform, rgrid, cgrid)
    xs = np.array(xs, dtype="float32").ravel()
    ys = np.array(ys, dtype="float32").ravel()

    flat_bands = [np.round(data[b].ravel(), round_decimals) for b in range(bands)]

    valid = ~np.isnan(flat_bands[0])
    if not np.any(valid):
        return None

    out = {"x": xs[valid], "y": ys[valid]}
    for b_name, fb in zip(band_names, flat_bands):
        out[b_name] = fb[valid]

    return out

def dict_to_table(col_dict, year, month):
    n = len(next(iter(col_dict.values())))
    year_col  = np.full(n, year  if year  is not None else -1, dtype="int16")
    month_col = np.full(n, month if month is not None else -1, dtype="int8")
    col_dict = {**col_dict, "year": year_col, "month": month_col}
    return pa.Table.from_pydict(col_dict)

# ============================================================
# TRACK YEARS
# ============================================================

all_years = set()
year_to_months = defaultdict(set)

# ============================================================
# TIME-VARYING DATASETS
# ============================================================

def process_timevarying_dataset(ds_cfg):
    name, kind = ds_cfg["name"], ds_cfg["kind"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    tifs = sorted([p for p in in_dir.glob("*.tif") if p.suffix == ".tif"])

    print(f"\n=== DATASET: {name} ({kind}) ===")

    # --- First, scan to find all years in this dataset ---
    years_in_dataset = set()
    for tif in tifs:
        y, m = extract_year_month(tif.name, kind)
        if y is not None:
            years_in_dataset.add(y)

    # --- Skip years that already have Parquet ---
    for year in sorted(years_in_dataset):
        out_path = out_dataset_dir / f"{name}_{year}.parquet"
        if out_path.exists():
            print(f"[SKIP already exists] {out_path}")
            continue

        print(f"  Processing YEAR {year}")

        tbl_list = []

        for tif in tifs:
            y, m = extract_year_month(tif.name, kind)
            if y != year:
                continue

            with rio.open(tif) as ds:
                band_names = get_band_names(ds, name)
                col_dict = clip_and_flatten(ds, study, band_names)
                if col_dict is None:
                    print(f"[SKIP no data in domain] {tif.name}")
                    continue

                if kind == "alt_annual":
                    # replicate 12 months
                    for month in range(1, 13):
                        year_to_months[y].add(month)
                        all_years.add(y)
                        tbl_list.append(dict_to_table(col_dict, y, month))
                else:
                    all_years.add(y)
                    year_to_months[y].add(m)
                    tbl_list.append(dict_to_table(col_dict, y, m))

        if tbl_list:
            final = pa.concat_tables(tbl_list)
            out_path = out_dataset_dir / f"{name}_{year}.parquet"
            pq.write_table(final, out_path, compression="snappy", use_dictionary=True)
            print(f"[WRITE] {out_path}")
        else:
            print(f"[WARN] No valid pixels for YEAR {year}")

# ============================================================
# STATIC DATASETS
# ============================================================

def process_static_dataset(ds_cfg):
    name = ds_cfg["name"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    if not all_years:
        print(f"[WARN] No dynamic datasets processed yet → skipping static {name}")
        return

    tifs = sorted([p for p in in_dir.glob("*.tif")])

    print(f"\n=== STATIC DATASET: {name} ===")

    for tif in tifs:
        with rio.open(tif) as ds:
            band_names = get_band_names(ds, name)
            base_cols = clip_and_flatten(ds, study, band_names)
            if base_cols is None:
                print(f"[SKIP static outside domain] {tif.name}")
                continue

            for year in sorted(all_years):
                out_path = out_dataset_dir / f"{name}_{year}.parquet"

                if out_path.exists():
                    print(f"[SKIP static exists] {out_path}")
                    continue

                tables = []
                for month in sorted(year_to_months[year]):
                    n = len(base_cols["x"])
                    col_dict = {
                        "x": base_cols["x"],
                        "y": base_cols["y"],
                        "year": np.full(n, year, dtype="int16"),
                        "month": np.full(n, month, dtype="int8"),
                    }
                    for b in band_names:
                        col_dict[b] = base_cols[b]
                    tables.append(pa.Table.from_pydict(col_dict))

                year_table = pa.concat_tables(tables)
                pq.write_table(year_table, out_path, compression="snappy", use_dictionary=True)
                print(f"[WRITE] {out_path}")

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # Pass 1: time-varying datasets
    for cfg in DATASETS:
        if cfg["kind"] != "static":
            process_timevarying_dataset(cfg)

    # Pass 2: static datasets
    for cfg in DATASETS:
        if cfg["kind"] == "static":
            process_static_dataset(cfg)

    print("\n[DONE] All datasets converted to yearly Parquet files.")



=== DATASET: ALT (alt_annual) ===
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1997.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1998.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1999.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2000.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2001.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2002.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2003.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2004.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2005.parquet
[SKIP alrea

In [1]:
't'

't'